# Day 12 · Imbalanced Data & Advanced Metrics

**Objective:** Remove 90% of diseased training images, inspect the effect on classification metrics, then use class-weighted `CrossEntropyLoss` to address the imbalance.

## 1. Download the dataset

We use the tomato classes from the same Kaggle dataset. Healthy training images are retained; only 10% of diseased training images are kept.

In [ ]:
!pip -q install kaggle
from google.colab import files
uploaded = files.upload()  # Upload kaggle.json
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.replace('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!mkdir -p /content/plant_data
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/plant_data --unzip

In [ ]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, models
from torchvision.transforms import v2
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
train_dirs = list(Path('/content/plant_data').rglob('train'))
valid_dirs = list(Path('/content/plant_data').rglob('valid'))
if not train_dirs or not valid_dirs:
    raise FileNotFoundError("Expected train and valid folders in the extracted dataset.")
train_dir, valid_dir = train_dirs[0], valid_dirs[0]

## 2. Select tomato classes and create class imbalance

All healthy images are retained. For every diseased tomato class, randomly keep only 10% of its training images. The validation set is left unchanged.

In [ ]:
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = v2.Compose([
    v2.Resize((224, 224)), v2.RandomHorizontalFlip(), v2.RandomRotation(15),
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean, std)
])
eval_tf = v2.Compose([
    v2.Resize((224, 224)), v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean, std)
])

train_all = datasets.ImageFolder(train_dir, transform=train_tf)
valid_all = datasets.ImageFolder(valid_dir, transform=eval_tf)
tomato_classes = [c for c in train_all.classes if c.startswith('Tomato___')]
class_to_new = {name: i for i, name in enumerate(tomato_classes)}

healthy_indices, diseased = [], {}
for i, (_, label) in enumerate(train_all.samples):
    name = train_all.classes[label]
    if name not in class_to_new:
        continue
    if 'healthy' in name.lower():
        healthy_indices.append(i)
    else:
        diseased.setdefault(name, []).append(i)

rng = np.random.default_rng(42)
kept_indices = healthy_indices.copy()
for name, indices in diseased.items():
    keep_n = max(1, int(0.10 * len(indices)))
    kept_indices.extend(rng.choice(indices, keep_n, replace=False).tolist())

class TomatoSubset(Dataset):
    def __init__(self, source, indices):
        self.source, self.indices = source, indices
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        image, old_label = self.source[self.indices[idx]]
        return image, class_to_new[self.source.classes[old_label]]

train_data = TomatoSubset(train_all, kept_indices)
valid_indices = [i for i, (_, label) in enumerate(valid_all.samples)
                 if valid_all.classes[label] in class_to_new]
valid_data = TomatoSubset(valid_all, valid_indices)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)

print('Healthy images retained:', len(healthy_indices))
print('Diseased images before:', sum(len(v) for v in diseased.values()))
print('Diseased images retained:', len(kept_indices) - len(healthy_indices))

## 3. Train an unweighted model

Start with ordinary cross-entropy. Review per-class precision, recall, and F1-score—not just accuracy. A model can score well overall while missing minority classes.

In [ ]:
def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, len(tomato_classes))
    return model.to(device)

def train_model(model, criterion, epochs=3):
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
        print(f'Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_data):.4f}')
    return model

def evaluate(model, title):
    model.eval()
    actual, predicted = [], []
    with torch.no_grad():
        for images, labels in valid_loader:
            outputs = model(images.to(device))
            predicted.extend(outputs.argmax(1).cpu().numpy())
            actual.extend(labels.numpy())
    print(title)
    print(classification_report(actual, predicted, labels=list(range(len(tomato_classes))),
                                target_names=tomato_classes, zero_division=0))
    return actual, predicted

baseline_model = train_model(build_model(), nn.CrossEntropyLoss())
y_true, y_pred = evaluate(baseline_model, 'Unweighted model')

## 4. Apply class-weighted CrossEntropyLoss

Weights are calculated from the number of retained training examples in each class. Rarer classes receive larger loss weights, so their mistakes contribute more to the training objective.

In [ ]:
class_counts = np.bincount(
    [class_to_new[train_all.classes[train_all.samples[i][1]]] for i in kept_indices],
    minlength=len(tomato_classes)
)
class_weights = len(kept_indices) / (len(tomato_classes) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

print('Class counts:', class_counts)
print('Class weights:', class_weights.cpu().numpy())

weighted_model = train_model(build_model(), nn.CrossEntropyLoss(weight=class_weights))
y_true_w, y_pred_w = evaluate(weighted_model, 'Weighted-loss model')

## 5. Compare confusion matrices

Compare the two models' confusion matrices and per-class recall. Weighting can improve minority-class recognition, but it does not guarantee improvement for every class.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, predictions, title in [
    (axes[0], y_pred, 'Unweighted loss'),
    (axes[1], y_pred_w, 'Weighted loss')
]:
    cm = confusion_matrix(y_true, predictions, labels=list(range(len(tomato_classes))))
    ConfusionMatrixDisplay(cm, display_labels=tomato_classes).plot(
        ax=ax, xticks_rotation=90, colorbar=False
    )
    ax.set_title(title)
plt.tight_layout()
plt.show()

### Day 12 · Key takeaway

Class imbalance can make accuracy misleading. Class-weighted cross-entropy increases the penalty for mistakes on underrepresented classes. Use per-class recall and the confusion matrix to judge whether it helped.